In [1]:
# ==============================================================================
# Simplified Keyword Extraction Pipeline: YAKE vs. KeyBERT
#
# This script compares two keyword extraction methods:
# 1. YAKE: A classical, statistical-based approach.
# 2. KeyBERT: A modern approach using a specialized biomedical AI model.


In [2]:
!pip install  pandas scikit-learn torch transformers datasets evaluate accelerate
!pip install yake keybert sentence-transformers rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.9/355.9 kB 12.1 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=abb1bb5fe78689a36925ba8084ec317f76012c1b5707ffa1bd774c53826bf3d1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [3]:
print("--- Step 1: Importing libraries ---")
import pandas as pd
import numpy as np
import re
import yake
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
import evaluate
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

--- Step 1: Importing libraries ---


In [4]:

# --- Step 2: Load and Preprocess Data ---
print("\n--- Step 2: Loading and preprocessing data ---")

df = pd.read_csv('mtsamples.csv')


df = df[['transcription', 'keywords']].dropna().reset_index(drop=True)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_keywords(keywords):
    return [kw.strip() for kw in str(keywords).split(',') if kw.strip()]



df['transcription_cleaned'] = df['transcription'].apply(clean_text)
df['keywords_parsed'] = df['keywords'].apply(parse_keywords)
df = df[df['keywords_parsed'].apply(len) > 0]

sample_size = 50
df_sample = df.sample(n=sample_size, random_state=42)
print(f"Using a sample of {len(df_sample)} records.")




--- Step 2: Loading and preprocessing data ---
Using a sample of 50 records.


In [5]:

print("\n--- Step 3: Initializing models ---")

# Model 1: YAKE
kw_extractor_yake = yake.KeywordExtractor(top=10, n=3)
def predict_yake(texts):
    print("\nRunning YAKE predictions...")
    return [[kw for kw, score in kw_extractor_yake.extract_keywords(text)] for text in tqdm(texts, desc="YAKE")]

# Model 2: KeyBERT with Specialized Model
print("Loading specialized SentenceTransformer model for KeyBERT...")
sentence_model = SentenceTransformer("pritamdeka/S-BioBert-snli-multinli-stsb")
kw_model_keybert = KeyBERT(model=sentence_model)

def predict_keybert_specialized(texts):
    print("\nRunning KeyBERT predictions with specialized model...")
    keywords_with_scores = kw_model_keybert.extract_keywords(texts, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=10)
    return [[kw for kw, score in kw_list] for kw_list in keywords_with_scores]



--- Step 3: Initializing models ---
Loading specialized SentenceTransformer model for KeyBERT...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
print("\n--- Step 4: Setting up evaluation functions ---")
rouge_scorer = evaluate.load('rouge')
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def run_evaluation(model_name, preds, labels):
    from sentence_transformers import util
    print(f"\nEvaluating {model_name}...")

    jaccard_scores = [len(set(p).intersection(set(l))) / len(set(p).union(set(l))) if len(set(p).union(set(l))) > 0 else 0 for p, l in zip(preds, labels)]
    pred_strings = [' '.join(p) for p in preds]
    label_strings = [' '.join(l) for l in labels]
    rouge_results = rouge_scorer.compute(predictions=pred_strings, references=label_strings)
    semantic_scores = []
    for pred, label in zip(preds, labels):
        if not pred or not label:
            semantic_scores.append(0)
            continue
        pred_emb = semantic_model.encode(' '.join(pred))
        label_emb = semantic_model.encode(' '.join(label))
        semantic_scores.append(util.cos_sim(pred_emb, label_emb).item())

    return {
        'Jaccard': np.mean(jaccard_scores),
        'ROUGE-1 (F1)': rouge_results['rouge1'],
        'ROUGE-L (F1)': rouge_results['rougeL'],
        'Semantic Similarity': np.mean(semantic_scores)
    }



--- Step 4: Setting up evaluation functions ---


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
if __name__ == "__main__":
    print("\n--- Step 5: Running models and evaluating performance ---")
    true_labels = df_sample['keywords_parsed'].tolist()
    test_texts = df_sample['transcription_cleaned'].tolist()

    yake_preds = predict_yake(test_texts)
    keybert_preds = predict_keybert_specialized(test_texts)

    results = {
        'YAKE': run_evaluation('YAKE', yake_preds, true_labels),
        'KeyBERT (Specialized)': run_evaluation('KeyBERT (Specialized)', keybert_preds, true_labels),
    }

    results_df = pd.DataFrame(results).T

    print("\n\n" + "="*25 + " FINAL RESULTS " + "="*25)
    print("Quantitative Model Performance Comparison")
    print(results_df)
    print("="*67)

    print("\n--- Example Predictions ---")
    for i in range(min(3, len(test_texts))):
        print(f"\n--- Example {i+1} ---")
        print(f"TRANSCRIPTION (snippet): {test_texts[i][:200]}...")
        print(f"TRUE KEYWORDS: {', '.join(true_labels[i])}")
        print(f"YAKE PREDICTION: {', '.join(yake_preds[i])}")
        print(f"KEYBERT (SPECIALIZED) PREDICTION: {', '.join(keybert_preds[i])}")
        print("-" * 20)

    print("\n--- Pipeline Finished ---")



--- Step 5: Running models and evaluating performance ---

Running YAKE predictions...


YAKE: 100%|██████████| 50/50 [00:02<00:00, 21.87it/s]



Running KeyBERT predictions with specialized model...

Evaluating YAKE...

Evaluating KeyBERT (Specialized)...


========================= FINAL RESULTS =========================
Quantitative Model Performance Comparison
                        Jaccard  ROUGE-1 (F1)  ROUGE-L (F1)  \
YAKE                   0.096803      0.300667      0.231114   
KeyBERT (Specialized)  0.076272      0.269522      0.203145   

                       Semantic Similarity  
YAKE                              0.603809  
KeyBERT (Specialized)             0.603418  

--- Example Predictions ---

--- Example 1 ---
TRANSCRIPTION (snippet): preoperative diagnosis: , chronic renal failure.,postoperative diagnosis: ,chronic renal failure.,procedure performed:, insertion of left femoral circle-c catheter.,anesthesia: , 1% lidocaine.,estimat...
TRUE KEYWORDS: nephrology, chronic renal failure, femoral circle-c catheter, indwelling catheter, catheter, insertion, seldinger, guidewire, indwelling, femoral, dialysis
YAKE 